In [1]:
# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path
import json
import copy
import warnings
import gc

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

print("Imports OK")

Imports OK


In [2]:
# ============================================================
# CONFIG
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility"
)

DATA_DIR = PROJECT_DIR / "processed_data_final"
RAW_DATA_DIR = PROJECT_DIR / "data"

OUTPUT_DIR = DATA_DIR / "extra_stress_test_lane_changes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Main asymmetric setting
BRAKE_THRESHOLD = -0.1
ACCEL_THRESHOLD = 0.2

HISTORY = 25
HORIZON = 1
STRIDE = 1
BATCH_SIZE = 512

SEQUENCE_TAG = f"H{HISTORY}_Hz{HORIZON}_S{STRIDE}"
IDM_FIRST_SEQUENCE_TAG = f"H{HISTORY}_S{STRIDE}"

THRESHOLD_TAG = (
    f"brake{abs(BRAKE_THRESHOLD):.1f}_accel{ACCEL_THRESHOLD:.1f}"
    .replace(".", "p")
)

IDM_PARAM_TAG = "refinedIDM_v2"

print("DEVICE:", DEVICE)
print("SEQUENCE_TAG:", SEQUENCE_TAG)
print("IDM_FIRST_SEQUENCE_TAG:", IDM_FIRST_SEQUENCE_TAG)
print("THRESHOLD_TAG:", THRESHOLD_TAG)
print("IDM_PARAM_TAG:", IDM_PARAM_TAG)
print("OUTPUT_DIR:", OUTPUT_DIR)

DEVICE: cuda
SEQUENCE_TAG: H25_Hz1_S1
IDM_FIRST_SEQUENCE_TAG: H25_S1
THRESHOLD_TAG: brake0p1_accel0p2
IDM_PARAM_TAG: refinedIDM_v2
OUTPUT_DIR: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\extra_stress_test_lane_changes


In [3]:
# ============================================================
# MODEL PATHS - DEPLOYABLE STRESS TEST MODELS
# ============================================================

# Plain Transformer
PLAIN_TRANSFORMER_DIR = (
    DATA_DIR
    / "transformer_baseline_runs"
    / f"transformer_baseline_{SEQUENCE_TAG}_{THRESHOLD_TAG}"
)

# Transformer + XGB-soft IDM feature
TRANSFORMER_IDM_FEATURE_DIR = (
    DATA_DIR
    / "transformer_idm_feature_runs"
    / f"TransformerIDMFeature_XGBSoftBestIDM_{SEQUENCE_TAG}_{THRESHOLD_TAG}_{IDM_PARAM_TAG}"
)

# IDM selector directory
IDM_SELECTOR_DIR = (
    DATA_DIR
    / "idm_selector_comparison_runs"
    / f"idm_selector_comparison_{THRESHOLD_TAG}"
)

# Residual Hybrid + Global IDM
HYBRID_GLOBAL_DIR = (
    DATA_DIR
    / "hybrid_idm_first_runs"
    / f"Hybrid_GlobalIDM_{IDM_FIRST_SEQUENCE_TAG}_{THRESHOLD_TAG}_{IDM_PARAM_TAG}"
)

# Optional weighted fusion with Global IDM
WEIGHTED_FUSION_GLOBAL_DIR = (
    DATA_DIR
    / "transformer_weighted_fusion_runs"
    / f"TransformerWeightedFusion_GlobalIDM_{SEQUENCE_TAG}_{THRESHOLD_TAG}_{IDM_PARAM_TAG}"
)

MODEL_DIRS_TO_CHECK = {
    "Plain Transformer": PLAIN_TRANSFORMER_DIR,
    "Transformer + XGB-soft best-IDM feature": TRANSFORMER_IDM_FEATURE_DIR,
    "IDM selector": IDM_SELECTOR_DIR,
    "Residual Hybrid + Global IDM": HYBRID_GLOBAL_DIR,
    "Optional Weighted Fusion + Global IDM": WEIGHTED_FUSION_GLOBAL_DIR,
}

print("=== MODEL DIRECTORIES ===")
for name, path in MODEL_DIRS_TO_CHECK.items():
    print(f"{name}:")
    print(path)
    print("Exists:", path.exists())
    print()

=== MODEL DIRECTORIES ===
Plain Transformer:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_baseline_runs\transformer_baseline_H25_Hz1_S1_brake0p1_accel0p2
Exists: True

Transformer + XGB-soft best-IDM feature:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_idm_feature_runs\TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p1_accel0p2_refinedIDM_v2
Exists: True

IDM selector:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\idm_selector_comparison_runs\idm_selector_comparison_brake0p1_accel0p2
Exists: True

Residual Hybrid + Global IDM:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\hybrid_idm_first_runs\Hybrid_GlobalIDM_H25_S1_brake0p1_accel0p2_refinedIDM_v2
Exists: True

Optional Weighted Fusion + Global IDM:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_weighted_fusion_runs\TransformerWeighte

In [4]:
# ============================================================
# LOAD ORIGINAL / RAW TRACKS
# ============================================================

# Option A: one combined raw file
possible_combined_files = [
    RAW_DATA_DIR / "tracks_all.csv",
    RAW_DATA_DIR / "raw_tracks_all.csv",
    DATA_DIR / "raw_tracks_all.csv",
    DATA_DIR / "all_tracks_raw.csv",
]

combined_file = None
for p in possible_combined_files:
    if p.exists():
        combined_file = p
        break

if combined_file is not None:
    print("Loading combined raw file:")
    print(combined_file)
    raw_tracks = pd.read_csv(combined_file)

else:
    # Option B: highD style files: xx_tracks.csv
    track_files = sorted(RAW_DATA_DIR.rglob("*_tracks.csv"))

    assert len(track_files) > 0, (
        f"No raw track files found in {RAW_DATA_DIR}. "
        "Set RAW_DATA_DIR correctly or provide a combined raw file."
    )

    dfs = []

    for p in track_files:
        print("Loading:", p.name)
        temp = pd.read_csv(p)

        # infer recording_id from filename if missing
        if "recording_id" not in temp.columns:
            stem = p.stem.replace("_tracks", "")
            try:
                rec_id = int(stem)
            except Exception:
                rec_id = stem
            temp["recording_id"] = rec_id

        dfs.append(temp)

    raw_tracks = pd.concat(dfs, ignore_index=True)

print("\nRaw tracks shape:", raw_tracks.shape)
print("Columns:")
print(raw_tracks.columns.tolist())

Loading: 01_tracks.csv
Loading: 02_tracks.csv
Loading: 03_tracks.csv
Loading: 04_tracks.csv
Loading: 05_tracks.csv
Loading: 06_tracks.csv
Loading: 07_tracks.csv
Loading: 08_tracks.csv
Loading: 09_tracks.csv
Loading: 10_tracks.csv
Loading: 11_tracks.csv
Loading: 12_tracks.csv
Loading: 13_tracks.csv
Loading: 14_tracks.csv
Loading: 15_tracks.csv
Loading: 16_tracks.csv
Loading: 17_tracks.csv
Loading: 18_tracks.csv
Loading: 19_tracks.csv
Loading: 20_tracks.csv
Loading: 21_tracks.csv
Loading: 22_tracks.csv
Loading: 23_tracks.csv
Loading: 24_tracks.csv
Loading: 25_tracks.csv
Loading: 26_tracks.csv
Loading: 27_tracks.csv
Loading: 28_tracks.csv
Loading: 29_tracks.csv
Loading: 30_tracks.csv
Loading: 31_tracks.csv
Loading: 32_tracks.csv
Loading: 33_tracks.csv
Loading: 34_tracks.csv
Loading: 35_tracks.csv
Loading: 36_tracks.csv
Loading: 37_tracks.csv
Loading: 38_tracks.csv
Loading: 39_tracks.csv
Loading: 40_tracks.csv
Loading: 41_tracks.csv
Loading: 42_tracks.csv
Loading: 43_tracks.csv
Loading: 44

In [5]:
# ============================================================
# BASIC COLUMN NORMALIZATION
# ============================================================

raw = raw_tracks.copy()

# Create recording_id if missing
if "recording_id" not in raw.columns:
    raw["recording_id"] = 0

# Create unique_id
if "unique_id" not in raw.columns:
    assert "id" in raw.columns, "Need either unique_id or id column."
    raw["unique_id"] = raw["recording_id"].astype(str) + "_" + raw["id"].astype(str)

required_cols = [
    "recording_id",
    "unique_id",
    "id",
    "frame",
    "laneId",
    "xVelocity",
    "xAcceleration",
    "precedingId",
    "dhw",
]

missing = [c for c in required_cols if c not in raw.columns]
assert len(missing) == 0, f"Missing required columns: {missing}"

raw = raw.sort_values(["unique_id", "frame"]).reset_index(drop=True)

print("Raw normalized shape:", raw.shape)
print(raw[required_cols].head())

Raw normalized shape: (39725708, 27)
   recording_id unique_id  id  frame  laneId  xVelocity  xAcceleration  \
0            10      10_1   1      1       2     -32.79          -0.29   
1            10      10_1   1      2       2     -32.80          -0.30   
2            10      10_1   1      3       2     -32.82          -0.30   
3            10      10_1   1      4       2     -32.83          -0.30   
4            10      10_1   1      5       2     -32.84          -0.30   

   precedingId  dhw  
0            0  0.0  
1            0  0.0  
2            0  0.0  
3            0  0.0  
4            0  0.0  


In [6]:
# ============================================================
# ADD / VERIFY LEADER SPEED
# ============================================================

if "precedingXVelocity" not in raw.columns:
    print("precedingXVelocity not found. Merging leader speed from raw tracks...")

    leader_lookup = raw[[
        "recording_id",
        "frame",
        "id",
        "xVelocity"
    ]].copy()

    leader_lookup = leader_lookup.rename(columns={
        "id": "precedingId",
        "xVelocity": "precedingXVelocity"
    })

    raw = raw.merge(
        leader_lookup,
        on=["recording_id", "frame", "precedingId"],
        how="left"
    )

else:
    print("precedingXVelocity already exists.")

print("Missing precedingXVelocity:", raw["precedingXVelocity"].isna().sum())
print(raw[["recording_id", "frame", "id", "precedingId", "xVelocity", "precedingXVelocity"]].head())

precedingXVelocity already exists.
Missing precedingXVelocity: 0
   recording_id  frame  id  precedingId  xVelocity  precedingXVelocity
0            10      1   1            0     -32.79                 0.0
1            10      2   1            0     -32.80                 0.0
2            10      3   1            0     -32.82                 0.0
3            10      4   1            0     -32.83                 0.0
4            10      5   1            0     -32.84                 0.0


In [7]:
# ============================================================
# BUILD STRESS FEATURES AND EVENT FLAGS WITH CACHE
# ============================================================

stress_features_path = OUTPUT_DIR / "stress_lane_change_features.csv"

FORCE_REBUILD_STRESS_FEATURES = False

if stress_features_path.exists() and not FORCE_REBUILD_STRESS_FEATURES:
    print("Loading cached stress features:")
    print(stress_features_path)

    stress_base = pd.read_csv(stress_features_path)

else:
    print("Building stress features from raw data...")

    df = raw.copy()

    # Basic car-following validity for first stress test
    df["leader_exists"] = df["precedingId"].fillna(0) > 0
    df["has_gap"] = df["dhw"].notna()
    df["has_leader_speed"] = df["precedingXVelocity"].notna()

    df["valid_cf_raw"] = (
        df["leader_exists"]
        & df["has_gap"]
        & df["has_leader_speed"]
    )

    # Features
    df["v_ego"] = df["xVelocity"].abs()
    df["v_leader"] = df["precedingXVelocity"].abs()
    df["delta_v"] = df["v_ego"] - df["v_leader"]
    df["gap"] = df["dhw"].clip(lower=0.1)
    df["a_true"] = df["xAcceleration"]

    # Remove invalid physical values for this first stress test
    stress_base = df[df["valid_cf_raw"]].copy()

    # Lane changers
    lane_counts = stress_base.groupby("unique_id")["laneId"].nunique()
    lane_changer_ids = lane_counts[lane_counts > 1].index

    stress_base["ego_lane_changer"] = stress_base["unique_id"].isin(lane_changer_ids)

    # Lane-change frame
    stress_base["lane_change_frame"] = (
        stress_base["laneId"] != stress_base.groupby("unique_id")["laneId"].shift(1)
    )

    # Leader change frame
    stress_base["leader_change_frame"] = (
        stress_base["precedingId"] != stress_base.groupby("unique_id")["precedingId"].shift(1)
    )

    # Windows around events
    def add_near_event_flag(df_in, event_col, out_col, window=25):
        out = df_in.copy()
        out[out_col] = False

        for uid, g in out.groupby("unique_id", sort=False):
            idx = g.index.to_numpy()
            event_positions = np.where(g[event_col].fillna(False).values)[0]

            near = np.zeros(len(g), dtype=bool)

            for pos in event_positions:
                lo = max(0, pos - window)
                hi = min(len(g), pos + window + 1)
                near[lo:hi] = True

            out.loc[idx, out_col] = near

        return out

    WINDOW = 25

    stress_base = add_near_event_flag(
        stress_base,
        event_col="lane_change_frame",
        out_col="near_lane_change",
        window=WINDOW
    )

    stress_base = add_near_event_flag(
        stress_base,
        event_col="leader_change_frame",
        out_col="near_leader_change",
        window=WINDOW
    )

    # Regime
    stress_base["regime"] = np.where(
        stress_base["a_true"] < BRAKE_THRESHOLD,
        "braking",
        np.where(stress_base["a_true"] > ACCEL_THRESHOLD, "accelerating", "steady")
    )

    # Save cache
    stress_base.to_csv(stress_features_path, index=False)

    print("Saved cached stress features:")
    print(stress_features_path)

print("Stress base shape:", stress_base.shape)

print("\nLane changer counts:")
print(stress_base["ego_lane_changer"].value_counts())

print("\nNear lane change counts:")
print(stress_base["near_lane_change"].value_counts())

print("\nNear leader change counts:")
print(stress_base["near_leader_change"].value_counts())

print("\nRegime counts:")
print(stress_base["regime"].value_counts())

Loading cached stress features:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\extra_stress_test_lane_changes\stress_lane_change_features.csv
Stress base shape: (31555155, 42)

Lane changer counts:
ego_lane_changer
False    28816619
True      2738536
Name: count, dtype: int64

Near lane change counts:
near_lane_change
False    28272806
True      3282349
Name: count, dtype: int64

Near leader change counts:
near_leader_change
False    27506088
True      4049067
Name: count, dtype: int64

Regime counts:
regime
steady          16004201
accelerating    10471435
braking          5079519
Name: count, dtype: int64


In [8]:
# ============================================================
# BUILD RAW STRESS EPISODES
# ============================================================

stress_base = stress_base.sort_values(["unique_id", "frame"]).copy()

# Break sequences when frame is not consecutive OR leader changes.
# For lane-change stress, we do NOT break on laneId changes; we want to evaluate near those.
frame_gap = stress_base["frame"] != stress_base.groupby("unique_id")["frame"].shift(1) + 1

# Option: keep leader changes inside sequences so model sees messier transitions.
# But for basic sequence continuity, break only on frame gaps.
stress_base["run_break"] = frame_gap.fillna(True)

stress_base["stress_episode_idx"] = stress_base.groupby("unique_id")["run_break"].cumsum()

stress_base["episode_id"] = (
    stress_base["unique_id"].astype(str)
    + "_stress_"
    + stress_base["stress_episode_idx"].astype(str)
)

# Keep only episodes long enough for history + horizon
episode_lengths = stress_base.groupby("episode_id").size()
valid_episodes = episode_lengths[episode_lengths >= HISTORY + HORIZON].index

stress_base = stress_base[stress_base["episode_id"].isin(valid_episodes)].copy()

print("Stress dataframe after episode filtering:", stress_base.shape)
print("Stress episodes:", stress_base["episode_id"].nunique())
print("Unique vehicles:", stress_base["unique_id"].nunique())

Stress dataframe after episode filtering: (31530866, 45)
Stress episodes: 110793
Unique vehicles: 108444


In [9]:
# ============================================================
# METRIC HELPERS
# ============================================================

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    corr = np.nan
    if len(y_true) > 1 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        corr = float(np.corrcoef(y_true, y_pred)[0, 1])

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MedAE": median_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Bias": float(np.mean(y_pred - y_true)),
        "Corr": corr,
        "Count": len(y_true),
    }


def subset_metrics(pred_df, model_name):
    rows = []

    subsets = {
        "All_CF_Raw": np.ones(len(pred_df), dtype=bool),
        "NonLaneChanger": ~pred_df["ego_lane_changer"].fillna(False).values,
        "LaneChangerTrack": pred_df["ego_lane_changer"].fillna(False).values,
        "NearLaneChange": pred_df["near_lane_change"].fillna(False).values,
        "NearLeaderChange": pred_df["near_leader_change"].fillna(False).values,
        "Braking": pred_df["regime"].values == "braking",
        "Braking_NearLaneChange": (
            (pred_df["regime"].values == "braking")
            & pred_df["near_lane_change"].fillna(False).values
        ),
        "Braking_NearLeaderChange": (
            (pred_df["regime"].values == "braking")
            & pred_df["near_leader_change"].fillna(False).values
        ),
    }

    for subset_name, mask in subsets.items():
        g = pred_df.loc[mask]

        if len(g) == 0:
            continue

        m = compute_metrics(g["y_true"], g["y_pred"])

        rows.append({
            "Model": model_name,
            "Subset": subset_name,
            **m,
            "UnderBrake_Rate_gt_0p5": np.mean((g["y_pred"] - g["y_true"]) > 0.5),
            "WrongSign_Rate": np.mean(np.sign(g["y_true"]) != np.sign(g["y_pred"])),
        })

    return pd.DataFrame(rows)

In [10]:
# ============================================================
# TRANSFORMER MODEL CLASSES
# ======================

import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)

        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerRegressor(nn.Module):
    def __init__(
        self,
        input_dim,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.1
    ):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=500)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        # IMPORTANT: checkpoint expects "transformer.layers..."
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # IMPORTANT: checkpoint expects "regressor..."
        self.regressor = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x_last = x[:, -1, :]
        return self.regressor(x_last)


class SequenceDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]


def predict_model(model, X, batch_size=BATCH_SIZE):
    dataset = SequenceDataset(X)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    preds = []

    model.eval()
    with torch.no_grad():
        for xb in loader:
            xb = xb.to(DEVICE)
            pred = model(xb).cpu().numpy()
            preds.append(pred)

    return np.vstack(preds)

In [11]:
# ============================================================
# BUILD TRANSFORMER STRESS SEQUENCES WITH CACHE
# ============================================================

BASE_FEATURE_COLS = ["v_ego", "v_leader", "delta_v", "gap"]

sequence_cache_path = OUTPUT_DIR / f"stress_transformer_sequences_{SEQUENCE_TAG}.npz"
meta_cache_path = OUTPUT_DIR / f"stress_transformer_meta_{SEQUENCE_TAG}.csv"

def make_transformer_sequences(df, feature_cols, history=HISTORY, horizon=HORIZON, stride=STRIDE):
    X_seq = []
    meta_rows = []

    for ep_id, g in df.groupby("episode_id", sort=False):
        g = g.sort_values("frame").reset_index(drop=True)

        values = g[feature_cols].values.astype(np.float32)

        for t in range(history - 1, len(g) - horizon, stride):
            hist_slice = slice(t - history + 1, t + 1)
            target_idx = t + horizon

            X_seq.append(values[hist_slice])

            row = {
                "episode_id": ep_id,
                "unique_id": g.loc[target_idx, "unique_id"],
                "recording_id": g.loc[target_idx, "recording_id"],
                "frame": g.loc[target_idx, "frame"],
                "y_true": g.loc[target_idx, "a_true"],
                "v_ego": g.loc[target_idx, "v_ego"],
                "v_leader": g.loc[target_idx, "v_leader"],
                "delta_v": g.loc[target_idx, "delta_v"],
                "gap": g.loc[target_idx, "gap"],
                "regime": g.loc[target_idx, "regime"],
                "ego_lane_changer": g.loc[target_idx, "ego_lane_changer"],
                "near_lane_change": g.loc[target_idx, "near_lane_change"],
                "near_leader_change": g.loc[target_idx, "near_leader_change"],
            }

            meta_rows.append(row)

    X_seq = np.asarray(X_seq, dtype=np.float32)
    meta_df = pd.DataFrame(meta_rows)

    assert len(X_seq) == len(meta_df), (
        f"Length mismatch: X={len(X_seq)}, meta={len(meta_df)}"
    )

    return X_seq, meta_df


if sequence_cache_path.exists() and meta_cache_path.exists():
    print("Loading cached stress sequences...")
    print(sequence_cache_path)
    print(meta_cache_path)

    cache = np.load(sequence_cache_path)
    X_stress_base = cache["X_stress_base"]
    meta_stress = pd.read_csv(meta_cache_path)

else:
    print("Building stress sequences...")

    X_stress_base, meta_stress = make_transformer_sequences(
        stress_base,
        BASE_FEATURE_COLS,
        HISTORY,
        HORIZON,
        STRIDE
    )

    np.savez_compressed(
        sequence_cache_path,
        X_stress_base=X_stress_base
    )

    meta_stress.to_csv(meta_cache_path, index=False)

    print("Saved cached stress sequences:")
    print(sequence_cache_path)
    print(meta_cache_path)

print("X_stress_base:", X_stress_base.shape)
print("meta_stress:", meta_stress.shape)

Loading cached stress sequences...
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\extra_stress_test_lane_changes\stress_transformer_sequences_H25_Hz1_S1.npz
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\extra_stress_test_lane_changes\stress_transformer_meta_H25_Hz1_S1.csv
X_stress_base: (28761041, 25, 4)
meta_stress: (28761041, 13)


In [12]:
# ============================================================
# STRESS TEST: PLAIN TRANSFORMER
# ============================================================

plain_output_path = OUTPUT_DIR / "stress_predictions_plain_transformer.csv"
plain_metrics_path = OUTPUT_DIR / "stress_metrics_plain_transformer.csv"

if plain_output_path.exists() and plain_metrics_path.exists():
    print("Loading existing Plain Transformer stress results...")
    plain_results = pd.read_csv(plain_output_path)
    plain_metrics = pd.read_csv(plain_metrics_path)

else:
    plain_ckpt_path = PLAIN_TRANSFORMER_DIR / "transformer_baseline_best.pt"
    plain_scaler_X_path = PLAIN_TRANSFORMER_DIR / "transformer_scaler_X.pkl"
    plain_scaler_y_path = PLAIN_TRANSFORMER_DIR / "transformer_scaler_y.pkl"

    assert plain_ckpt_path.exists(), f"Missing checkpoint: {plain_ckpt_path}"
    assert plain_scaler_X_path.exists(), f"Missing scaler: {plain_scaler_X_path}"
    assert plain_scaler_y_path.exists(), f"Missing scaler: {plain_scaler_y_path}"

    plain_scaler_X = joblib.load(plain_scaler_X_path)
    plain_scaler_y = joblib.load(plain_scaler_y_path)

    # Scale sequence
    n, h, f = X_stress_base.shape
    X_flat = X_stress_base.reshape(-1, f)
    X_scaled_flat = plain_scaler_X.transform(X_flat)
    X_stress_base_s = X_scaled_flat.reshape(n, h, f).astype(np.float32)

    checkpoint = torch.load(plain_ckpt_path, map_location=DEVICE)

    state_dict = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint

    # Build config safely
    model_config = checkpoint.get("model_config", {})
    model_config = {
        "input_dim": model_config.get("input_dim", X_stress_base_s.shape[-1]),
        "d_model": model_config.get("d_model", 64),
        "nhead": model_config.get("nhead", 4),
        "num_layers": model_config.get("num_layers", 2),
        "dim_feedforward": model_config.get("dim_feedforward", 128),
        "dropout": model_config.get("dropout", 0.1),
    }

    print("Model config:")
    print(model_config)

    plain_model = TransformerRegressor(**model_config).to(DEVICE)
    plain_model.load_state_dict(state_dict)

    plain_pred_s = predict_model(plain_model, X_stress_base_s)
    plain_pred = plain_scaler_y.inverse_transform(plain_pred_s).reshape(-1)

    plain_results = meta_stress.copy()
    plain_results["y_pred"] = plain_pred
    plain_results["error"] = plain_results["y_pred"] - plain_results["y_true"]
    plain_results["abs_error"] = plain_results["error"].abs()
    plain_results["Model"] = "Plain Transformer stress"

    plain_results.to_csv(plain_output_path, index=False)

    plain_metrics = subset_metrics(plain_results, "Plain Transformer stress")
    plain_metrics.to_csv(plain_metrics_path, index=False)

print("=== PLAIN TRANSFORMER STRESS METRICS ===")
print(plain_metrics.to_string(index=False))

Loading existing Plain Transformer stress results...
=== PLAIN TRANSFORMER STRESS METRICS ===
                   Model                   Subset      MAE     RMSE    MedAE        R2      Bias     Corr    Count  UnderBrake_Rate_gt_0p5  WrongSign_Rate
Plain Transformer stress               All_CF_Raw 0.144283 0.270863 0.072729  0.286601 -0.015426 0.552573 28761041                0.033341        0.230161
Plain Transformer stress           NonLaneChanger 0.138470 0.259470 0.070654  0.321140 -0.014452 0.574931 26296462                0.031511        0.227013
Plain Transformer stress         LaneChangerTrack 0.206310 0.371257 0.102814  0.024254 -0.025819 0.426497  2464579                0.052872        0.263751
Plain Transformer stress           NearLaneChange 0.301213 0.533546 0.140500 -0.969285 -0.041134 0.206568   541518                0.077630        0.336541
Plain Transformer stress         NearLeaderChange 0.251495 0.461159 0.109371 -0.679102 -0.051317 0.232523  1258816                0

In [13]:
# ============================================================
# IDM FUNCTIONS AND FINAL ASYMMETRIC PARAMETERS
# ============================================================

def idm_acceleration(v, delta_v, s, params):
    v0 = max(float(params["v0"]), 1e-6)
    T = float(params["T"])
    a = max(float(params["a"]), 1e-6)
    b = max(float(params["b"]), 1e-6)
    s0 = float(params["s0"])
    delta = float(params["delta"])

    v = np.maximum(np.asarray(v), 0.0)
    s = np.maximum(np.asarray(s), 0.1)

    sqrt_ab = np.sqrt(a * b)

    s_star = s0 + np.maximum(
        0.0,
        v * T + (v * delta_v) / (2.0 * sqrt_ab)
    )

    acc = a * (1.0 - (v / v0) ** delta - (s_star / s) ** 2)
    return acc


# Final asymmetric refined IDM parameters
best_global_params = {
    "v0": 42.0,
    "T": 0.5,
    "s0": 0.5,
    "a": 0.4,
    "b": 5.0,
    "delta": 4.0,
}

best_params_braking = {
    "v0": 1.0,
    "T": 0.3,
    "s0": 5.0,
    "a": 0.05,
    "b": 26.0,
    "delta": 0.5,
}

best_params_accel = {
    "v0": 115.0,
    "T": 0.1,
    "s0": 0.05,
    "a": 0.4,
    "b": 22.0,
    "delta": 14.0,
}

best_params_steady = {
    "v0": 44.0,
    "T": 0.3,
    "s0": 0.1,
    "a": 0.1,
    "b": 8.0,
    "delta": 4.0,
}

print("Using final asymmetric refined IDM parameters.")
print("Global:", best_global_params)
print("Braking:", best_params_braking)
print("Accelerating:", best_params_accel)
print("Steady:", best_params_steady)

Using final asymmetric refined IDM parameters.
Global: {'v0': 42.0, 'T': 0.5, 's0': 0.5, 'a': 0.4, 'b': 5.0, 'delta': 4.0}
Braking: {'v0': 1.0, 'T': 0.3, 's0': 5.0, 'a': 0.05, 'b': 26.0, 'delta': 0.5}
Accelerating: {'v0': 115.0, 'T': 0.1, 's0': 0.05, 'a': 0.4, 'b': 22.0, 'delta': 14.0}
Steady: {'v0': 44.0, 'T': 0.3, 's0': 0.1, 'a': 0.1, 'b': 8.0, 'delta': 4.0}


In [14]:
# ============================================================
# STRESS TEST: GLOBAL IDM
# ============================================================

global_idm_output_path = OUTPUT_DIR / "stress_predictions_global_idm.csv"
global_idm_metrics_path = OUTPUT_DIR / "stress_metrics_global_idm.csv"

if global_idm_output_path.exists() and global_idm_metrics_path.exists():
    print("Loading existing Global IDM stress results...")
    global_idm_results = pd.read_csv(global_idm_output_path)
    global_idm_metrics = pd.read_csv(global_idm_metrics_path)

else:
    global_idm_results = meta_stress.copy()

    global_idm_results["y_pred"] = idm_acceleration(
        global_idm_results["v_ego"],
        global_idm_results["delta_v"],
        global_idm_results["gap"],
        best_global_params
    ).clip(-8, 4)

    global_idm_results["error"] = global_idm_results["y_pred"] - global_idm_results["y_true"]
    global_idm_results["abs_error"] = global_idm_results["error"].abs()
    global_idm_results["Model"] = "Global IDM stress"

    global_idm_results.to_csv(global_idm_output_path, index=False)

    global_idm_metrics = subset_metrics(global_idm_results, "Global IDM stress")
    global_idm_metrics.to_csv(global_idm_metrics_path, index=False)

print("=== GLOBAL IDM STRESS METRICS ===")
print(global_idm_metrics.to_string(index=False))

Loading existing Global IDM stress results...
=== GLOBAL IDM STRESS METRICS ===
            Model                   Subset      MAE     RMSE    MedAE         R2      Bias      Corr    Count  UnderBrake_Rate_gt_0p5  WrongSign_Rate
Global IDM stress               All_CF_Raw 0.355122 0.625966 0.242508  -2.810096 -0.010846 -0.069146 28761041                0.072688        0.342551
Global IDM stress           NonLaneChanger 0.331493 0.541407 0.239340  -1.955652  0.020185 -0.083249 26296462                0.072651        0.335150
Global IDM stress         LaneChangerTrack 0.607238 1.202110 0.291850  -9.230022 -0.341933 -0.022064  2464579                0.073077        0.421509
Global IDM stress           NearLaneChange 0.892619 1.850584 0.306989 -22.690938 -0.638424  0.004509   541518                0.064847        0.419523
Global IDM stress         NearLeaderChange 0.610548 1.347525 0.267981 -13.336713 -0.307170 -0.014411  1258816                0.062199        0.350173
Global IDM stress   

In [15]:
# ============================================================
# STRESS TEST: XGB-SOFT BEST-IDM SELECTOR
# Creates idm_stress for later Transformer + IDM feature stress
# ============================================================

xgb_idm_output_path = OUTPUT_DIR / "stress_predictions_xgb_soft_best_idm.csv"
xgb_idm_metrics_path = OUTPUT_DIR / "stress_metrics_xgb_soft_best_idm.csv"

FORCE_REBUILD_XGB_STRESS = True

if FORCE_REBUILD_XGB_STRESS:
    for p in [xgb_idm_output_path, xgb_idm_metrics_path]:
        if p.exists():
            p.unlink()
            print("Deleted old file:", p)

if xgb_idm_output_path.exists() and xgb_idm_metrics_path.exists():
    print("Loading existing XGB-soft best-IDM stress results...")
    idm_stress = pd.read_csv(xgb_idm_output_path)
    xgb_idm_metrics = pd.read_csv(xgb_idm_metrics_path)

else:
    # ========================================================
    # 1. Find saved XGB selector bundle
    # ========================================================

    selector_bundle_candidates = [
        # Asymmetric bundle saved by Transformer + IDM feature notebook
        TRANSFORMER_IDM_FEATURE_DIR / "xgb_soft_best_idm_selector_bundle.joblib",

        # Possible locations in selector folder
        IDM_SELECTOR_DIR / "xgb_soft_best_idm_selector_bundle.joblib",
        IDM_SELECTOR_DIR / "xgb_soft_best_idm_selector_bundle.pkl",
        IDM_SELECTOR_DIR / "xgb_best_selector_bundle.pkl",
        IDM_SELECTOR_DIR / "xgb_best_idm_selector_bundle.pkl",
        IDM_SELECTOR_DIR / "xgb_selector_bundle.pkl",

        # Fallbacks
        DATA_DIR / "xgb_soft_best_idm_selector_bundle.joblib",
        DATA_DIR / "xgb_soft_best_idm_selector_bundle.pkl",
        DATA_DIR / "xgb_best_selector_bundle.pkl",
    ]

    selector_bundle_path = None

    for p in selector_bundle_candidates:
        if p.exists():
            selector_bundle_path = p
            break

    assert selector_bundle_path is not None, (
        "Missing XGB selector bundle. Checked:\n"
        + "\n".join(str(p) for p in selector_bundle_candidates)
    )

    print("Using XGB selector bundle:")
    print(selector_bundle_path)

    selector_bundle = joblib.load(selector_bundle_path)

    print("Bundle keys:")
    if isinstance(selector_bundle, dict):
        print(selector_bundle.keys())
    else:
        print(type(selector_bundle))

    # ========================================================
    # 2. Extract XGB model and selector columns
    # ========================================================

    if isinstance(selector_bundle, dict):
        xgb_model = (
            selector_bundle.get("model", None)
            or selector_bundle.get("xgb_model", None)
            or selector_bundle.get("selector_model", None)
            or selector_bundle.get("best_model", None)
        )

        le_best = (
            selector_bundle.get("label_encoder", None)
            or selector_bundle.get("le_best", None)
            or selector_bundle.get("encoder", None)
        )

        selector_cols = (
            selector_bundle.get("selector_cols", None)
            or selector_bundle.get("feature_cols", None)
            or selector_bundle.get("features", None)
        )
    else:
        xgb_model = selector_bundle
        le_best = None
        selector_cols = None

    assert xgb_model is not None, (
        "Could not find XGB model inside selector bundle."
    )

    if selector_cols is None:
        selector_cols = [
            "v_ego",
            "v_leader",
            "delta_v",
            "gap",
            "time_headway",
            "closing_speed",
            "opening_speed",
            "abs_delta_v",
            "inv_ttc",
            "log_gap",
        ]

    print("Selector columns:")
    print(selector_cols)

    # ========================================================
    # 3. Use meta_stress as base
    # ========================================================
    # meta_stress is already aligned with Transformer stress sequences.
    # This avoids the length mismatch with full stress_base.
    # ========================================================

    assert "meta_stress" in globals(), (
        "meta_stress does not exist. Run the transformer stress sequence cell first."
    )

    idm_stress = meta_stress.copy().reset_index(drop=True)

    # Some notebooks call this column 'regime' instead of 'true_regime'
    if "true_regime" not in idm_stress.columns and "regime" in idm_stress.columns:
        idm_stress["true_regime"] = idm_stress["regime"]

    print("idm_stress rows:", len(idm_stress))

    # ========================================================
    # 4. Candidate IDM predictions
    # ========================================================

    idm_stress["a_idm_global"] = idm_acceleration(
        idm_stress["v_ego"],
        idm_stress["delta_v"],
        idm_stress["gap"],
        best_global_params,
    ).clip(-8, 4)

    idm_stress["a_idm_braking"] = idm_acceleration(
        idm_stress["v_ego"],
        idm_stress["delta_v"],
        idm_stress["gap"],
        best_params_braking,
    ).clip(-8, 4)

    idm_stress["a_idm_accel"] = idm_acceleration(
        idm_stress["v_ego"],
        idm_stress["delta_v"],
        idm_stress["gap"],
        best_params_accel,
    ).clip(-8, 4)

    idm_stress["a_idm_steady"] = idm_acceleration(
        idm_stress["v_ego"],
        idm_stress["delta_v"],
        idm_stress["gap"],
        best_params_steady,
    ).clip(-8, 4)

    # ========================================================
    # 5. Selector features
    # ========================================================

    idm_stress["time_headway"] = (
        idm_stress["gap"] / np.maximum(idm_stress["v_ego"], 0.1)
    )

    idm_stress["closing_speed"] = np.maximum(idm_stress["delta_v"], 0.0)
    idm_stress["opening_speed"] = np.maximum(-idm_stress["delta_v"], 0.0)
    idm_stress["abs_delta_v"] = np.abs(idm_stress["delta_v"])
    idm_stress["inv_ttc"] = (
        idm_stress["closing_speed"] / np.maximum(idm_stress["gap"], 0.1)
    )
    idm_stress["log_gap"] = np.log1p(np.minimum(idm_stress["gap"], 500.0))

    missing_selector_cols = [
        c for c in selector_cols if c not in idm_stress.columns
    ]

    assert len(missing_selector_cols) == 0, (
        f"Missing selector columns: {missing_selector_cols}"
    )

    X_selector = idm_stress[selector_cols].copy()
    X_selector = X_selector.replace([np.inf, -np.inf], np.nan)
    X_selector = X_selector.fillna(0.0)

    # ========================================================
    # 6. XGB soft prediction
    # ========================================================

    proba = xgb_model.predict_proba(X_selector)

    if le_best is not None:
        try:
            classes = list(le_best.inverse_transform(xgb_model.classes_))
        except Exception:
            classes = list(le_best.classes_)
    else:
        classes = list(xgb_model.classes_)

    print("XGB selector classes:")
    print(classes)

    def normalize_class_name(cls):
        cls = str(cls).lower()

        if cls in ["global", "global_idm"]:
            return "global"

        if cls in ["braking", "brake"]:
            return "braking"

        if cls in ["accelerating", "acceleration", "accel"]:
            return "accelerating"

        if cls in ["steady", "steady_state"]:
            return "steady"

        return cls

    class_to_col = {
        "global": "a_idm_global",
        "braking": "a_idm_braking",
        "accelerating": "a_idm_accel",
        "steady": "a_idm_steady",
    }

    idm_stress["y_pred"] = 0.0

    for j, cls in enumerate(classes):
        cls_norm = normalize_class_name(cls)

        assert cls_norm in class_to_col, (
            f"Unknown XGB class '{cls}'. Normalized as '{cls_norm}'. "
            f"Expected one of {list(class_to_col.keys())}."
        )

        prob_col = f"p_{cls_norm}"
        idm_col = class_to_col[cls_norm]

        idm_stress[prob_col] = proba[:, j]
        idm_stress["y_pred"] += proba[:, j] * idm_stress[idm_col].values

    idm_stress["y_pred"] = idm_stress["y_pred"].clip(-8, 4)

    # ========================================================
    # 7. Metrics and save
    # ========================================================

    idm_stress["error"] = idm_stress["y_pred"] - idm_stress["y_true"]
    idm_stress["abs_error"] = idm_stress["error"].abs()
    idm_stress["Model"] = "XGB-soft best-IDM selector stress"

    idm_stress.to_csv(xgb_idm_output_path, index=False)

    xgb_idm_metrics = subset_metrics(
        idm_stress,
        "XGB-soft best-IDM selector stress",
    )

    xgb_idm_metrics.to_csv(xgb_idm_metrics_path, index=False)

print("=== XGB-SOFT BEST-IDM SELECTOR STRESS METRICS ===")
print(xgb_idm_metrics.to_string(index=False))

Deleted old file: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\extra_stress_test_lane_changes\stress_predictions_xgb_soft_best_idm.csv
Deleted old file: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\extra_stress_test_lane_changes\stress_metrics_xgb_soft_best_idm.csv
Using XGB selector bundle:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_idm_feature_runs\TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p1_accel0p2_refinedIDM_v2\xgb_soft_best_idm_selector_bundle.joblib
Bundle keys:
dict_keys(['model', 'label_encoder', 'selector_cols', 'classes', 'idm_param_tag', 'threshold_tag', 'sequence_tag'])
Selector columns:
['v_ego', 'v_leader', 'delta_v', 'gap', 'time_headway', 'closing_speed', 'opening_speed', 'abs_delta_v', 'inv_ttc', 'log_gap']
idm_stress rows: 28761041
XGB selector classes:
['accelerating', 'braking', 'global', 'steady']
=== XGB-SOFT BEST-IDM SELECTOR STRESS METRI

In [16]:
# --------------------------------------------------------
# Add XGB-soft IDM prior to stress data
# Use idm_stress directly because it is already sequence-aligned
# --------------------------------------------------------

feature_stress_base = idm_stress.copy().reset_index(drop=True)

print("feature_stress_base rows:", len(feature_stress_base))

assert "y_pred" in feature_stress_base.columns, (
    "idm_stress must contain the XGB-soft IDM prediction in column 'y_pred'."
)

feature_stress_base["a_idm_prior"] = feature_stress_base["y_pred"].values

missing_prior = feature_stress_base["a_idm_prior"].isna().sum()
print("Rows missing a_idm_prior:", missing_prior)

assert missing_prior == 0, "Some rows are missing XGB-soft IDM prior."

feature_stress_base rows: 28761041
Rows missing a_idm_prior: 0


In [17]:
# ============================================================
# STRESS TEST: HYBRID GLOBAL IDM
# ============================================================

# Build global IDM prior on stress data
hybrid_stress_base = stress_base.copy()

hybrid_stress_base["a_idm_prior"] = idm_acceleration(
    hybrid_stress_base["v_ego"],
    hybrid_stress_base["delta_v"],
    hybrid_stress_base["gap"],
    best_global_params
).clip(-8, 4)

hybrid_stress_base["residual_true"] = hybrid_stress_base["a_true"] - hybrid_stress_base["a_idm_prior"]

HYBRID_FEATURE_COLS = [
    "v_ego",
    "v_leader",
    "delta_v",
    "gap",
    "a_idm_prior"
]

X_hybrid, meta_hybrid = make_transformer_sequences(
    hybrid_stress_base,
    HYBRID_FEATURE_COLS,
    HISTORY,
    HORIZON,
    STRIDE
)

# Add a_idm_prior to meta
meta_hybrid = meta_hybrid.merge(
    hybrid_stress_base[["episode_id", "frame", "a_idm_prior"]],
    on=["episode_id", "frame"],
    how="left"
)

# Load scalers
hybrid_scaler_path = HYBRID_GLOBAL_DIR / "scalers.npz"
assert hybrid_scaler_path.exists(), f"Missing hybrid scalers: {hybrid_scaler_path}"

scaler_data = np.load(hybrid_scaler_path)

feature_mean = scaler_data["feature_mean"]
feature_std = scaler_data["feature_std"]
target_mean = float(scaler_data["target_mean"])
target_std = float(scaler_data["target_std"])

X_hybrid_s = ((X_hybrid - feature_mean) / feature_std).astype(np.float32)

# Load hybrid checkpoint
hybrid_ckpt_candidates = [
    HYBRID_GLOBAL_DIR / "best_model.pt",
    HYBRID_GLOBAL_DIR / "best_checkpoint.pt",
    HYBRID_GLOBAL_DIR / "last_checkpoint.pt",
]

hybrid_ckpt_path = None
for p in hybrid_ckpt_candidates:
    if p.exists():
        hybrid_ckpt_path = p
        break

assert hybrid_ckpt_path is not None, f"No hybrid checkpoint found in {HYBRID_GLOBAL_DIR}"

print("Using hybrid checkpoint:")
print(hybrid_ckpt_path)

hybrid_checkpoint = torch.load(hybrid_ckpt_path, map_location=DEVICE)

# If checkpoint is only state_dict
if isinstance(hybrid_checkpoint, dict) and "model_state_dict" in hybrid_checkpoint:
    hybrid_state = hybrid_checkpoint["model_state_dict"]
    hybrid_model_config = hybrid_checkpoint.get("model_config", {
        "input_dim": X_hybrid_s.shape[-1],
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 128,
        "dropout": 0.1,
    })
else:
    hybrid_state = hybrid_checkpoint
    hybrid_model_config = {
        "input_dim": X_hybrid_s.shape[-1],
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 128,
        "dropout": 0.1,
    }

hybrid_model = TransformerRegressor(**hybrid_model_config).to(DEVICE)

# ------------------------------------------------------------
# Load checkpoint with key-name compatibility
# Old checkpoint uses: encoder / head
# Current class uses: transformer / regressor
# ------------------------------------------------------------

def remap_hybrid_state_dict(state_dict):
    new_state = {}

    for k, v in state_dict.items():
        new_k = k

        # Old model naming -> current model naming
        if new_k.startswith("encoder."):
            new_k = new_k.replace("encoder.", "transformer.", 1)

        if new_k.startswith("head."):
            new_k = new_k.replace("head.", "regressor.", 1)

        new_state[new_k] = v

    return new_state


# If checkpoint is saved as a dict with model_state_dict, extract it
if isinstance(hybrid_state, dict) and "model_state_dict" in hybrid_state:
    hybrid_state = hybrid_state["model_state_dict"]

elif isinstance(hybrid_state, dict) and "state_dict" in hybrid_state:
    hybrid_state = hybrid_state["state_dict"]

# Remap old checkpoint keys
hybrid_state = remap_hybrid_state_dict(hybrid_state)

# Load with strict=False because pos_encoder.pe may not be saved in older checkpoints
load_info = hybrid_model.load_state_dict(hybrid_state, strict=False)

print("Hybrid model load info:")
print(load_info)


hybrid_residual_pred_s = predict_model(hybrid_model, X_hybrid_s).reshape(-1)
hybrid_residual_pred = hybrid_residual_pred_s * target_std + target_mean

hybrid_pred = meta_hybrid["a_idm_prior"].values + hybrid_residual_pred

hybrid_results = meta_hybrid.copy()
hybrid_results["y_pred"] = hybrid_pred
hybrid_results["error"] = hybrid_results["y_pred"] - hybrid_results["y_true"]
hybrid_results["abs_error"] = hybrid_results["error"].abs()
hybrid_results["Model"] = "Hybrid Global IDM stress"

hybrid_results.to_csv(
    OUTPUT_DIR / "stress_predictions_hybrid_global_idm.csv",
    index=False
)

hybrid_metrics = subset_metrics(hybrid_results, "Hybrid Global IDM stress")

print("=== HYBRID GLOBAL IDM STRESS METRICS ===")
print(hybrid_metrics.to_string(index=False))

Using hybrid checkpoint:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\hybrid_idm_first_runs\Hybrid_GlobalIDM_H25_S1_brake0p1_accel0p2_refinedIDM_v2\best_model.pt
Hybrid model load info:
_IncompatibleKeys(missing_keys=['pos_encoder.pe'], unexpected_keys=[])
=== HYBRID GLOBAL IDM STRESS METRICS ===
                   Model                   Subset      MAE     RMSE    MedAE        R2      Bias      Corr    Count  UnderBrake_Rate_gt_0p5  WrongSign_Rate
Hybrid Global IDM stress               All_CF_Raw 0.272975 0.376042 0.209638 -0.375015 -0.172666  0.128182 28761041                0.022455        0.662561
Hybrid Global IDM stress           NonLaneChanger 0.265858 0.362151 0.206077 -0.322468 -0.168288  0.150065 26296462                0.021613        0.664547
Hybrid Global IDM stress         LaneChangerTrack 0.348909 0.500817 0.259539 -0.775608 -0.219377  0.066009  2464579                0.031432        0.641367
Hybrid Global IDM stress           NearLaneCh

In [18]:
# ============================================================
# STRESS TEST: WEIGHTED FUSION + GLOBAL IDM
# Uses final asymmetric validation-selected fusion weight
# ============================================================

fusion_output_path = OUTPUT_DIR / "stress_predictions_weighted_fusion_global_idm.csv"
fusion_metrics_path = OUTPUT_DIR / "stress_metrics_weighted_fusion_global_idm.csv"

FORCE_REBUILD_FUSION_STRESS = True

if FORCE_REBUILD_FUSION_STRESS:
    for p in [fusion_output_path, fusion_metrics_path]:
        if p.exists():
            p.unlink()
            print("Deleted old file:", p)

if fusion_output_path.exists() and fusion_metrics_path.exists():
    print("Loading existing Weighted Fusion + Global IDM stress results...")
    fusion_global_results = pd.read_csv(fusion_output_path)
    fusion_global_metrics = pd.read_csv(fusion_metrics_path)

else:
    assert "plain_results" in globals(), "Run Plain Transformer stress cell first."
    assert "global_idm_results" in globals(), "Run Global IDM stress cell first."

    # Final asymmetric validation-selected weight:
    # Transformer weight = 0.95, Global IDM weight = 0.05
    alpha_transformer = 0.95
    alpha_idm = 0.05

    # Align by row order after checking lengths
    plain_aligned = plain_results.copy().reset_index(drop=True)
    idm_aligned = global_idm_results.copy().reset_index(drop=True)

    assert len(plain_aligned) == len(idm_aligned), (
        f"Length mismatch: plain_results has {len(plain_aligned)} rows, "
        f"global_idm_results has {len(idm_aligned)} rows."
    )

    fusion_global_results = plain_aligned.copy()

    fusion_global_results["y_pred"] = (
        alpha_transformer * plain_aligned["y_pred"].values
        + alpha_idm * idm_aligned["y_pred"].values
    )

    fusion_global_results["error"] = (
        fusion_global_results["y_pred"] - fusion_global_results["y_true"]
    )
    fusion_global_results["abs_error"] = fusion_global_results["error"].abs()
    fusion_global_results["Model"] = "Weighted Fusion + Global IDM stress"

    fusion_global_results.to_csv(fusion_output_path, index=False)

    fusion_global_metrics = subset_metrics(
        fusion_global_results,
        "Weighted Fusion + Global IDM stress",
    )

    fusion_global_metrics.to_csv(fusion_metrics_path, index=False)

print("=== WEIGHTED FUSION + GLOBAL IDM STRESS METRICS ===")
print(fusion_global_metrics.to_string(index=False))

=== WEIGHTED FUSION + GLOBAL IDM STRESS METRICS ===
                              Model                   Subset      MAE     RMSE    MedAE        R2      Bias     Corr    Count  UnderBrake_Rate_gt_0p5  WrongSign_Rate
Weighted Fusion + Global IDM stress               All_CF_Raw 0.146152 0.270457 0.073989  0.288737 -0.015197 0.547670 28761041                0.032695        0.233397
Weighted Fusion + Global IDM stress           NonLaneChanger 0.140016 0.259463 0.071447  0.321177 -0.012721 0.570540 26296462                0.031199        0.229673
Weighted Fusion + Global IDM stress         LaneChangerTrack 0.211631 0.367840 0.111445  0.042131 -0.041625 0.422927  2464579                0.048664        0.273134
Weighted Fusion + Global IDM stress           NearLaneChange 0.303569 0.518258 0.154032 -0.858048 -0.070999 0.208668   541518                0.067359        0.341244
Weighted Fusion + Global IDM stress         NearLeaderChange 0.251599 0.450311 0.113996 -0.601033 -0.064109 0.230790  

In [19]:
# ============================================================
# COMBINE FINAL STRESS TEST RESULTS
# ============================================================

stress_metric_frames = []

if "global_idm_metrics" in globals():
    stress_metric_frames.append(global_idm_metrics)

if "xgb_idm_metrics" in globals():
    stress_metric_frames.append(xgb_idm_metrics)

if "plain_metrics" in globals():
    stress_metric_frames.append(plain_metrics)

if "hybrid_metrics" in globals():
    stress_metric_frames.append(hybrid_metrics)

if "fusion_global_metrics" in globals():
    stress_metric_frames.append(fusion_global_metrics)

assert len(stress_metric_frames) > 0, "No stress metrics found."

all_stress_metrics = pd.concat(stress_metric_frames, ignore_index=True)

stress_summary_path = OUTPUT_DIR / "stress_test_deployable_summary.csv"
all_stress_metrics.to_csv(stress_summary_path, index=False)

print("=== FINAL DEPLOYABLE STRESS TEST SUMMARY ===")
print(all_stress_metrics.sort_values("MAE").to_string(index=False))

print("\nSaved stress summary to:")
print(stress_summary_path)

=== FINAL DEPLOYABLE STRESS TEST SUMMARY ===
                              Model                   Subset      MAE     RMSE    MedAE         R2      Bias      Corr    Count  UnderBrake_Rate_gt_0p5  WrongSign_Rate
           Plain Transformer stress           NonLaneChanger 0.138470 0.259470 0.070654   0.321140 -0.014452  0.574931 26296462                0.031511        0.227013
Weighted Fusion + Global IDM stress           NonLaneChanger 0.140016 0.259463 0.071447   0.321177 -0.012721  0.570540 26296462                0.031199        0.229673
           Plain Transformer stress               All_CF_Raw 0.144283 0.270863 0.072729   0.286601 -0.015426  0.552573 28761041                0.033341        0.230161
Weighted Fusion + Global IDM stress               All_CF_Raw 0.146152 0.270457 0.073989   0.288737 -0.015197  0.547670 28761041                0.032695        0.233397
  XGB-soft best-IDM selector stress           NonLaneChanger 0.204559 0.316533 0.131709  -0.010286 -0.029685  0.126